# Paper B batch 4 — SQuAD v1.1 (standard extractive QA), QA vs BIO across tokenizer families (6.21)
Input is `"<question> <answer sentence>"` (trainers tokenize one string); the answer's character span is the gold. QA and BIO see identical input. Encoders: XLM-R (SentencePiece), mBERT (WordPiece), RoBERTa + ModernBERT (byte-BPE), 3 seeds, joint vs BIO; BIO at the heuristic O-weight and at `O_WEIGHT=1.0`. Scored with run_08 (original / extend / strip gold). The official SQuAD EM strips punctuation, so it is blind to the artifact; strict character EM is not.
Runtime → **A100** (T4 works, slower). Cell 4 is a 1-epoch timing dry-run: read its wall time before launching cells 5–6. Every launch skips runs whose `metrics.json` already exists on Drive, so a disconnect is safe — just rerun. **After any code change set `FORCE=1` for the affected block.** Last cell releases the runtime.

In [ ]:
# 1. Setup: Drive + repo + deps + GPU gate
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf /content/repo && git clone https://github.com/JustLetMeBeHello/Idiomator_Research.git /content/repo
%cd /content/repo
!git log --oneline -1
!pip install -q -r Requirements.txt datasets
import torch
assert torch.cuda.is_available(), 'CPU-only runtime (happens after restarts) — Runtime > Change runtime type > GPU'
print('Device: cuda —', torch.cuda.get_device_name(0))

In [ ]:
# 2. Preflight: patch pushed, Drive roots durable + writable. T0 proves runs are fresh.
import os, json, glob, time, shutil, subprocess
T0 = time.time()
assert os.path.exists('experiments/rigor/run_38_squad_prep.py'), 'old commit — push run_38 first'
M = '/content/drive/MyDrive'
SQ, SQ_OW1 = f'{M}/IdiomatorRigor_NER_squad', f'{M}/IdiomatorRigor_NER_squad_ow1'
# (HF id, short, trust_remote_code, OUT_SUB) — OUT_SUB is where run_08 SOURCES looks for each encoder
ENCS = [('xlm-roberta-base','xlmr',0,'flip'), ('bert-base-multilingual-cased','mbert',0,'flip'),
        ('roberta-base','roberta',0,'bbpe'), ('answerdotai/ModernBERT-base','modernbert',1,'bbpe')]
for r in (SQ, SQ_OW1):
    os.makedirs(r, exist_ok=True)
    assert os.path.realpath(r).startswith('/content/drive'), f'{r} is not on Drive'
    open(f'{r}/.probe','w').write('ok'); assert open(f'{r}/.probe').read()=='ok'; os.remove(f'{r}/.probe')
def readback(root, pat='*/*_s*/test_predictions.jsonl', fresh_only=False):
    for f in sorted(glob.glob(f'{root}/{pat}')):
        mt = os.path.getmtime(f)
        if fresh_only and mt < T0: continue
        print(f'{sum(1 for _ in open(f)):5d} rows  {time.ctime(mt)}  {f[len(root)+1:]}')
print('preflight ok')

In [ ]:
# 3. Build SQuAD sentence-context splits (~5 min: random pages from the HF datasets-server). Precondition: answers are often followed by punctuation.
r = subprocess.run(['python','experiments/rigor/run_38_squad_prep.py','--out-dir','data/squad_sent/Splits','--force'],capture_output=True,text=True)
print(r.stdout[-700:], r.stderr[-500:])
meta = json.load(open('data/squad_sent/Splits/meta.json')); OW = meta['o_weight']
assert [meta['splits'][s]['n'] for s in ('train','dev','test')] == [3000,500,500]
assert meta['splits']['test']['punct_follow_pct'] > 20, 'precondition missing — answers not followed by punctuation'
print('O_WEIGHT (heuristic):', OW)

In [ ]:
# 4. Timing dry-run (1 epoch, both trainers) for the slowest and the fastest-to-break encoders. NOTE the wall times to size cells 5-6.
for model, short, trc, sub in (ENCS[0], ENCS[3]):
    t = time.time()
    !DRIVE_OUT={SQ} DATA_DIR=data/squad_sent/Splits LANGS=English TEST_LANGS=English O_WEIGHT={OW} LR_JOINT=3e-5 LR_BIO=3e-5 WORK_LOCAL=1 TRUST_REMOTE_CODE={trc} OUT_SUB={sub} MODEL={model} ENC_SHORT={short} bash experiments/rigor/run_07_mdeberta_sp_replication.sh --dry-run
    print(f'### {short} dry-run (1 epoch joint + 1 epoch bio): {time.time()-t:.0f}s  — a full run is ~7x/6x that per system')
    !rm -rf {SQ}/{sub}/_dryrun_joint {SQ}/{sub}/_dryrun_bio

In [ ]:
# 5. Launch original-weight matrix: 4 encoders x seeds 42 123 7 x {joint, bio} = 24 runs. Resumable.
FORCE = 0
for model, short, trc, sub in ENCS:
    !DRIVE_OUT={SQ} DATA_DIR=data/squad_sent/Splits LANGS=English TEST_LANGS=English O_WEIGHT={OW} LR_JOINT=3e-5 LR_BIO=3e-5 WORK_LOCAL=1 TRUST_REMOTE_CODE={trc} FORCE={FORCE} SEEDS="42 123 7" OUT_SUB={sub} MODEL={model} ENC_SHORT={short} bash experiments/rigor/run_07_mdeberta_sp_replication.sh

In [ ]:
# 6. ow1: copy the joint runs just produced (weight-independent), train BIO at O_WEIGHT=1.0 (12 runs). Resumable.
FORCE = 0
for model, short, trc, sub in ENCS:
    for s in (42,123,7):
        src, dst = f'{SQ}/{sub}/{short}_joint_s{s}', f'{SQ_OW1}/{sub}/{short}_joint_s{s}'
        assert os.path.exists(f'{src}/metrics.json'), f'missing joint source {src} — cell 5 incomplete'
        if not os.path.exists(f'{dst}/metrics.json'): shutil.copytree(src, dst, dirs_exist_ok=True)
    !DRIVE_OUT={SQ_OW1} DATA_DIR=data/squad_sent/Splits LANGS=English TEST_LANGS=English O_WEIGHT=1.0 LR_JOINT=3e-5 LR_BIO=3e-5 WORK_LOCAL=1 TRUST_REMOTE_CODE={trc} FORCE={FORCE} SEEDS="42 123 7" OUT_SUB={sub} MODEL={model} ENC_SHORT={short} bash experiments/rigor/run_07_mdeberta_sp_replication.sh

In [ ]:
# 7. Read back (joint rows == bio rows == 500) then score each root with NEW output names.
for tag, root in (('orig', SQ), ('ow1', SQ_OW1)):
    print(f'\n##### squad [{tag}]'); readback(root)
    for _, short, _, sub in ENCS:
        for s in (42,123,7):
            nj = sum(1 for _ in open(f'{root}/{sub}/{short}_joint_s{s}/test_predictions.jsonl')); nb = sum(1 for _ in open(f'{root}/{sub}/{short}_bio_s{s}/test_predictions.jsonl'))
            assert nj == nb == 500, f'{tag} {short} s{s}: joint {nj} vs bio {nb} rows (expect 500)'
    r = subprocess.run(['python','experiments/rigor/run_08_extended_gold.py','--preds-root',root,'--seeds','42','123','7','--out',f'{root}/run_08_squad_{tag}_3seed.json','--force'],capture_output=True,text=True,env={**os.environ,'BBPE':'1'})
    print(r.stdout[-3500:], r.stderr[-1200:])

In [ ]:
# FINAL: prove everything is on Drive, flush, then release the runtime.
# "Run all" stops at the first error, so this only runs if every cell above succeeded.
missing = []
for root in (SQ, SQ_OW1):
    for _, short, _, sub in ENCS:
        for sysn in ('joint','bio'):
            for s in (42,123,7): missing.append(f'{root}/{sub}/{short}_{sysn}_s{s}')
missing = [d for d in missing if not (os.path.exists(f'{d}/metrics.json') and os.path.getsize(f'{d}/test_predictions.jsonl') > 0)]
assert not missing, f'{len(missing)} runs not persisted to Drive, NOT disconnecting: {missing[:5]}'
print('✓ all expected runs persisted on Drive')
drive.flush_and_unmount()   # sync pending Drive writes before the VM disappears
print('Drive flushed — releasing runtime')
from google.colab import runtime
runtime.unassign()